In [ ]:
# ============================================================
# EXPERIMENT 08: INTELLIGENT CUSTOMER SUPPORT SYSTEM
# SaaS Domain | Human-in-the-Loop Control System
# CELL 1: SYSTEM IMPLEMENTATION
# ============================================================

import csv, json, uuid, re, time
from datetime import datetime, timedelta, timezone
from collections import Counter
from IPython.display import display
import pandas as pd

# -------------------- CONFIGURATION --------------------

CONFIG = {
    "sla_minutes": {"P1": 15, "P2": 60, "P3": 240, "P4": 1440},
    "confidence_threshold": 70,
    "retry_attempts": 3,
    "retry_delay": 0
}

# -------------------- SAMPLE CUSTOMER DATA --------------------

queries = [
    ["Q001", "C101", "I cannot log in to my account", "2026-11-03 10:00"],
    ["Q002", "C102", "Where can I find my invoice?", "2026-11-03 10:05"],
    ["Q003", "C103", "Application is down for all users", "2026-11-03 10:10"],
    ["Q004", "C104", "Please add dark mode feature", "2026-11-03 10:15"],
    ["Q005", "C105", "", "2026-11-03 10:20"],
    ["Q006", "C106", "It is not working", "2026-11-03 10:25"],
    ["Q007", "C107", "My payment failed and billing is incorrect", "2026-11-03 10:30"],
    ["Q008", "C108", "Export produces an unknown server error", "2026-11-03 10:35"],
    ["Q009", "C109", "How do I configure a custom dashboard?", "2026-11-03 10:40"],
    ["Q010", "C110", "I suspect a security incident and data breach", "2026-11-03 10:45"],
    ["Q011", "C111", "My account password needs recovery", "2026-11-03 10:50"],
    ["Q012", "C112", "Subscription payment is not reflected", "2026-11-03 10:55"],
    ["Q001", "C113", "Another query with duplicate ID", "2026-11-03 11:00"]
]

FIELDS = ["query_id", "customer_id", "query_text", "created_at"]

with open("customer_queries.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(FIELDS)
    writer.writerows(queries)

# -------------------- APPROVED KNOWLEDGE BASE --------------------

KB = [
    {"article_id": "KB001", "intent": "Login / Access",
     "keywords": ["login", "password", "account"],
     "approved": True,
     "response": "Use the approved account recovery procedure to reset your password. Contact support if access remains blocked."},

    {"article_id": "KB002", "intent": "Billing",
     "keywords": ["invoice", "payment", "billing"],
     "approved": True,
     "response": "Check the billing portal and invoice history. Contact billing support if payment is not reflected."},

    {"article_id": "KB003", "intent": "General Information",
     "keywords": ["plan", "features", "dashboard"],
     "approved": True,
     "response": "Refer to the approved product documentation for plan features and dashboard configuration."},

    {"article_id": "KB004", "intent": "Technical Error",
     "keywords": ["export", "server error", "application"],
     "approved": True,
     "response": "Record the affected functionality and error details. Technical support will investigate unresolved errors."}
]

with open("knowledge_base.json", "w") as f:
    json.dump(KB, f, indent=4)

In [ ]:



# -------------------- AUDIT AND LOG STORAGE --------------------

tickets = []
audit_log = []
notification_log = []
rejected_records = []
seen_ids = set()

def log_event(query_id, ticket_id, event, details=""):
    audit_log.append({
        "timestamp": datetime.now(timezone.utc).isoformat(),
        "query_id": query_id,
        "ticket_id": ticket_id,
        "event": event,
        "details": details
    })

# -------------------- QUERY INGESTION AND VALIDATION --------------------

def ingest_queries():
    valid = []

    with open("customer_queries.csv", newline="") as f:
        reader = csv.DictReader(f)

        for row in reader:
            row = {k: (v or "").strip() for k, v in row.items()}
            errors = []

            for field in FIELDS:
                if not row.get(field):
                    errors.append("Missing/empty " + field)

            qid = row.get("query_id", "")

            if qid and qid in seen_ids:
                errors.append("Duplicate query ID")
            elif qid:
                seen_ids.add(qid)

            if errors:
                rejected_records.append({
                    "query_id": qid,
                    "errors": "; ".join(errors)
                })
                log_event(qid, None, "QUERY_REJECTED", str(errors))
            else:
                valid.append(row)

    return valid

# -------------------- CLASSIFICATION ENGINE --------------------

def classify_query(query):
    text = query.lower()

    # Critical conditions have highest precedence
    if any(x in text for x in [
        "security incident", "data breach",
        "application is down", "all users",
        "service outage"
    ]):
        return {"intent": "Technical Error",
                "complexity": "High", "priority": "P1"}

    if any(x in text for x in [
        "not working", "did not work",
        "still unresolved", "unknown error"
    ]):
        return {"intent": "Unresolved Issue",
                "complexity": "High", "priority": "P2"}

    if any(x in text for x in [
        "invoice", "payment", "billing", "subscription"
    ]):
        return {"intent": "Billing",
                "complexity": "Medium", "priority": "P2"}

    if any(x in text for x in [
        "log in", "login", "password", "account locked"
    ]):
        return {"intent": "Login / Access",
                "complexity": "Medium", "priority": "P2"}

    if any(x in text for x in [
        "dark mode", "feature request", "add feature"
    ]):
        return {"intent": "Feature Request",
                "complexity": "Low", "priority": "P4"}

    if any(x in text for x in [
        "dashboard", "basic plan", "product information"
    ]):
        return {"intent": "General Information",
                "complexity": "Low", "priority": "P3"}

    return {"intent": "General Information",
            "complexity": "Medium", "priority": "P3"}

# -------------------- KNOWLEDGE RETRIEVAL --------------------

def retrieve_article(query):
    matches = []

    for article in kb_articles:
        if article["status"] != "APPROVED":
            continue

        keywords = article["keywords"]
        match_count = sum(
            1 for keyword in keywords
            if keyword.lower() in query.lower()
        )

        if match_count > 0:
            score = (match_count / len(keywords)) * 100
            matches.append((score, article))

    if not matches:
        return 0, None

    score, article = max(matches, key=lambda item: item[0])
    return score, article

# -------------------- WORKFLOW ROUTER --------------------

def route_query(classification, score, article):
    priority = classification["priority"]
    complexity = classification["complexity"]

    if priority == "P1":
        return "URGENT_HUMAN_ESCALATION"

    if complexity == "High":
        return "TECHNICAL_HUMAN_REVIEW"

    if article is None:
        return "CREATE_TICKET_AND_ESCALATE"

    if score >= CONFIG["confidence_threshold"] and complexity == "Low":
        return "AUTO_RESOLUTION"

    if score >= 40:
        return "HUMAN_REVIEW"

    return "REQUEST_INFORMATION_OR_ESCALATE"

# -------------------- TICKET MANAGEMENT --------------------

def create_ticket(query, classification, route):
    now = datetime.now(timezone.utc)
    priority = classification["priority"]

    ticket = {
        "ticket_id": "TKT-" + uuid.uuid4().hex[:8].upper(),
        "query_id": query["query_id"],
        "customer_id": query["customer_id"],
        "intent": classification["intent"],
        "complexity": classification["complexity"],
        "priority": priority,
        "route": route,
        "status": "OPEN",
        "assigned_team": None,
        "created_at": now.isoformat(),
        "sla_deadline": (
            now + timedelta(minutes=CONFIG["sla_minutes"][priority])
        ).isoformat(),
        "resolution": "",
        "reviewer_action": ""
    }

    tickets.append(ticket)
    log_event(query["query_id"], ticket["ticket_id"],
              "TICKET_CREATED", route)

    return ticket

# -------------------- HUMAN-IN-THE-LOOP MODULE --------------------

ALLOWED_ACTIONS = {
    "APPROVE", "EDIT", "HOLD",
    "REQUEST_INFORMATION", "ESCALATE", "RESOLVE"
}

def human_review(ticket, action, reviewer="AGENT_01", edited_response=None):
    if action not in ALLOWED_ACTIONS:
        raise ValueError("Unauthorized or invalid review action")

    ticket["reviewer_action"] = action

    states = {
        "APPROVE": "APPROVED_FOR_RESPONSE",
        "EDIT": "APPROVED_FOR_RESPONSE",
        "HOLD": "ON_HOLD",
        "REQUEST_INFORMATION": "AWAITING_CUSTOMER",
        "ESCALATE": "ESCALATED",
        "RESOLVE": "RESOLVED"
    }

    if action == "EDIT" and edited_response:
        ticket["resolution"] = edited_response

    ticket["status"] = states[action]
    ticket["reviewer"] = reviewer

    if action in ("APPROVE", "EDIT", "RESOLVE"):
        if ticket["status"] == "APPROVED_FOR_RESPONSE":
            ticket["status"] = "RESOLVED"

    log_event(ticket["query_id"], ticket["ticket_id"],
              "HUMAN_" + action, reviewer)

    return ticket

# -------------------- NOTIFICATION SERVICE --------------------

def send_notification(ticket, event, fail_first=0):
    message_id = "MSG-" + uuid.uuid4().hex[:8].upper()

    templates = {
        "CREATED": "Your support ticket has been created.",
        "ESCALATED": "Your issue has been escalated to specialist support.",
        "RESOLVED": "Your support ticket has been resolved.",
        "SLA_BREACH": "A support ticket has breached its SLA."
    }

    message = templates.get(event, "Your ticket has been updated.")
    success = False

    for attempt in range(1, CONFIG["retry_attempts"] + 1):
        # Simulate temporary delivery failure
        delivered = attempt > fail_first

        notification_log.append({
            "message_id": message_id,
            "ticket_id": ticket["ticket_id"],
            "channel": "Email + Messaging",
            "event": event,
            "attempt": attempt,
            "status": "SENT" if delivered else "FAILED",
            "message": message
        })

        log_event(ticket["query_id"], ticket["ticket_id"],
                  "NOTIFICATION_ATTEMPT",
                  f"{event}, attempt={attempt}, delivered={delivered}")

        if delivered:
            success = True
            break

    return "SENT" if success else "FAILED"

# -------------------- SLA MONITOR --------------------

def check_sla(ticket, current_time=None):
    now = current_time or datetime.now(timezone.utc)
    deadline = datetime.fromisoformat(ticket["sla_deadline"])

    if ticket["status"] in ("RESOLVED", "CLOSED"):
        return "COMPLETED"

    if now >= deadline:
        status = "BREACHED"
    elif deadline - now <= timedelta(minutes=5):
        status = "AT_RISK"
    else:
        status = "WITHIN_SLA"

    if status in ("AT_RISK", "BREACHED"):
        log_event(ticket["query_id"], ticket["ticket_id"],
                  "SLA_" + status)

    return status

# -------------------- COMPLETE WORKFLOW --------------------

def process_query(query):
    classification = classify_query(query["query_text"])

    score, article = retrieve_article(query["query_text"])

    route = route_query(classification, score, article)

    result = {
        "query_id": query["query_id"],
        "intent": classification["intent"],
        "complexity": classification["complexity"],
        "priority": classification["priority"],
        "confidence": score,
        "matched_article": article["article_id"] if article else "None",
        "route": route
    }

    if route == "AUTO_RESOLUTION":
        result["ticket_id"] = "AUTO-" + query["query_id"]
        result["status"] = "RESOLVED"
        send_notification(
            query["query_id"],
            "Your query has been automatically resolved."
        )
    else:
        ticket = create_ticket(query, classification, route)
        result["ticket_id"] = ticket["ticket_id"]
        result["status"] = ticket["status"]

    return result

# -------------------- CELL 1 COMPLETION --------------------

print("INTELLIGENT CUSTOMER SUPPORT SYSTEM INITIALIZED")
print("SaaS knowledge base loaded:", len(KB), "approved articles")
print("Sample queries generated:", len(queries))
print("Classification, HITL, ticketing, notification and SLA modules ready.")


INTELLIGENT CUSTOMER SUPPORT SYSTEM INITIALIZED
SaaS knowledge base loaded: 4 approved articles
Sample queries generated: 13
Classification, HITL, ticketing, notification and SLA modules ready.


In [ ]:
# ============================================================
# EXPERIMENT 08: INTELLIGENT CUSTOMER SUPPORT SYSTEM
# CELL 2: EXECUTION, HUMAN REVIEW AND EVALUATION
# ============================================================

import json
import pandas as pd
from datetime import datetime, timezone, timedelta

print("=" * 65)
print("INTELLIGENT CUSTOMER SUPPORT SYSTEM - EXECUTION")
print("=" * 65)

# -------------------- 0. LOAD KNOWLEDGE BASE --------------------

with open("knowledge_base.json", "r") as f:
    kb_data = json.load(f)

# Handle either a direct list or a dictionary containing articles
if isinstance(kb_data, list):
    knowledge_base = kb_data
elif isinstance(kb_data, dict):
    knowledge_base = next(
        (v for v in kb_data.values()
         if isinstance(v, list) and v and isinstance(v[0], dict)
         and "keywords" in v[0]),
        []
    )
else:
    knowledge_base = []

if not knowledge_base:
    raise ValueError(
        "Could not load knowledge-base articles. "
        "Check the structure of knowledge_base.json."
    )

print("Approved knowledge-base articles:", len(knowledge_base))

# -------------------- 1. CORRECT RETRIEVAL FUNCTION --------------------

def retrieve_article(query):
    matches = []

    for article in knowledge_base:
        if article.get("status", "").upper() != "APPROVED":
            continue

        keywords = article.get("keywords", [])

        match_count = sum(
            1 for keyword in keywords
            if keyword.lower() in query.lower()
        )

        if match_count > 0 and keywords:
            score = (match_count / len(keywords)) * 100
            matches.append((score, article))

    if not matches:
        return 0, None

    score, article = max(matches, key=lambda item: item[0])
    return score, article


# -------------------- 2. RESET EXECUTION STATE --------------------

tickets.clear()
audit_log.clear()
notification_log.clear()
rejected_records.clear()
seen_ids.clear()

# -------------------- 3. QUERY INGESTION --------------------

valid_queries = ingest_queries()

print("\n1. QUERY INGESTION")
print("Total records:", len(queries))
print("Valid records:", len(valid_queries))
print("Rejected records:", len(rejected_records))

# -------------------- 4. PROCESS WORKFLOW --------------------

results = []

for query in valid_queries:
    results.append(process_query(query))

results_df = pd.DataFrame(results)

print("\n2. CLASSIFICATION AND WORKFLOW ROUTING")
display(results_df)

# -------------------- 5. HUMAN-IN-THE-LOOP REVIEW --------------------

print("\n3. HUMAN-IN-THE-LOOP REVIEW")

review_actions = {
    "Q001": "APPROVE",
    "Q002": "EDIT",
    "Q004": "HOLD",
    "Q008": "ESCALATE",
    "Q009": "RESOLVE"
}

for qid, action in review_actions.items():

    ticket = next(
        (t for t in tickets if t["query_id"] == qid), None
    )

    if ticket and ticket["status"] not in (
        "RESOLVED", "ESCALATED"
    ):

        edited = (
            "Please follow the updated approved troubleshooting instructions."
            if action == "EDIT" else None
        )

        human_review(ticket, action, "AGENT_01", edited)

        if action in ("APPROVE", "EDIT", "RESOLVE"):
            send_notification(ticket, "RESOLVED")

        elif action == "ESCALATE":
            send_notification(ticket, "ESCALATED")

        print(qid, "->", action, "->", ticket["status"])

# -------------------- 6. NOTIFICATION RETRY TEST --------------------

print("\n4. NOTIFICATION RETRY TEST")

if tickets:
    retry_ticket = tickets[0]

    retry_status = send_notification(
        retry_ticket, "CREATED", fail_first=2
    )

    print("Simulated failures: 2")
    print("Final delivery status:", retry_status)
    print("Total attempts:", CONFIG["retry_attempts"])

    failure_status = send_notification(
        retry_ticket, "ESCALATED", fail_first=3
    )

    print("Permanent failure test:", failure_status)

else:
    print("Notification test skipped: No tickets available.")

# -------------------- 7. SLA MONITORING --------------------

print("\n5. SLA MONITORING")

if len(tickets) >= 2:

    sla_ticket = tickets[0]
    sla_ticket["status"] = "IN_REVIEW"
    sla_ticket["sla_deadline"] = (
        datetime.now(timezone.utc) + timedelta(minutes=3)
    ).isoformat()

    at_risk = check_sla(sla_ticket)
    print("At-risk test:", at_risk)

    breach_ticket = tickets[1]
    breach_ticket["status"] = "IN_REVIEW"
    breach_ticket["sla_deadline"] = (
        datetime.now(timezone.utc) - timedelta(minutes=10)
    ).isoformat()

    breach_status = check_sla(breach_ticket)
    print("Breach test:", breach_status)

    if breach_status == "BREACHED":
        send_notification(breach_ticket, "SLA_BREACH")

else:
    at_risk = "NOT_TESTED"
    breach_status = "NOT_TESTED"
    print("SLA tests skipped: Insufficient tickets.")

# -------------------- 8. AUTHORIZATION TEST --------------------

print("\n6. HUMAN REVIEW AUTHORIZATION TEST")

if tickets:
    try:
        human_review(
            tickets[0], "DELETE", "UNKNOWN_AGENT"
        )

    except ValueError as error:
        print("Invalid action rejected:", error)

        log_event(
            tickets[0]["query_id"],
            tickets[0]["ticket_id"],
            "UNAUTHORIZED_ACTION_REJECTED"
        )

# -------------------- 9. FINAL TICKET REPORT --------------------

ticket_columns = [
    "ticket_id", "query_id", "intent",
    "priority", "route", "assigned_team",
    "status", "reviewer_action"
]

ticket_df = pd.DataFrame(tickets, columns=ticket_columns)

print("\n7. FINAL TICKET LIFECYCLE REPORT")
display(ticket_df)

# -------------------- 10. NOTIFICATION REPORT --------------------

notification_df = pd.DataFrame(notification_log)

print("\n8. NOTIFICATION DELIVERY REPORT")
display(notification_df)

# -------------------- 11. AUDIT LOG --------------------

audit_df = pd.DataFrame(audit_log)

print("\n9. AUDIT LOG")
display(audit_df.tail(15))

# -------------------- 12. INVALID RECORD REPORT --------------------

print("\n10. INVALID RECORD REPORT")

rejected_df = pd.DataFrame(rejected_records)
display(rejected_df)

# -------------------- 13. EVALUATION METRICS --------------------

total_valid = len(valid_queries)
total_rejected = len(rejected_records)

ground_truth = {
    "Q001": "Login / Access",
    "Q002": "Billing",
    "Q003": "Technical Error",
    "Q004": "Feature Request",
    "Q006": "Unresolved Issue",
    "Q007": "Billing",
    "Q008": "Technical Error",
    "Q009": "General Information",
    "Q010": "Technical Error",
    "Q011": "Login / Access",
    "Q012": "Billing"
}

correct = sum(
    1 for row in results
    if ground_truth.get(row["query_id"]) == row["intent"]
)

classification_accuracy = (
    correct / total_valid * 100 if total_valid else 0
)

eligible = sum(
    1 for row in results
    if row["route"] == "AUTO_RESOLUTION"
)

resolved = sum(
    1 for t in tickets
    if t["route"] == "AUTO_RESOLUTION"
    and t["status"] == "RESOLVED"
)

resolution_rate = (
    resolved / eligible * 100 if eligible else 0
)

escalated = sum(
    1 for t in tickets
    if t["status"] in (
        "ESCALATED", "IN_REVIEW",
        "ON_HOLD", "AWAITING_CUSTOMER"
    )
)

escalation_rate = (
    escalated / total_valid * 100 if total_valid else 0
)

if not notification_df.empty:
    delivery_requests = notification_df["message_id"].nunique()

    delivered = sum(
        1 for _, group in notification_df.groupby("message_id")
        if (group["status"] == "SENT").any()
    )
else:
    delivery_requests = 0
    delivered = 0

notification_success = (
    delivered / delivery_requests * 100
    if delivery_requests else 0
)

sla_results = [
    x for x in [at_risk, breach_status]
    if x != "NOT_TESTED"
]

sla_compliance = (
    sum(x in ("WITHIN_SLA", "AT_RISK") for x in sla_results)
    / len(sla_results) * 100
    if sla_results else 0
)

retry_count = sum(
    1 for row in notification_log
    if row["attempt"] > 1
)

metrics = pd.DataFrame([
    ["Intent Classification Accuracy", f"{classification_accuracy:.2f}%"],
    ["Knowledge Base Resolution Rate", f"{resolution_rate:.2f}%"],
    ["Human Escalation / Review Rate", f"{escalation_rate:.2f}%"],
    ["Notification Success Rate", f"{notification_success:.2f}%"],
    ["SLA Compliance", f"{sla_compliance:.2f}%"],
    ["Valid Queries", total_valid],
    ["Rejected Queries", total_rejected],
    ["Total Tickets Created", len(tickets)],
    ["Notification Requests", delivery_requests],
    ["Retried Notification Attempts", retry_count]
], columns=["Evaluation Metric", "Result"])

print("\n11. SYSTEM EVALUATION SUMMARY")
display(metrics)

# -------------------- 14. SAVE OUTPUT LOGS --------------------

audit_df.to_csv("audit_log.csv", index=False)
notification_df.to_csv("notification_log.csv", index=False)
ticket_df.to_csv("tickets.csv", index=False)
rejected_df.to_csv("rejected_queries.csv", index=False)

print("\n" + "=" * 65)
print("EXPERIMENT 08 EXECUTION COMPLETED")
print("=" * 65)

print("Generated files:")
print("1. customer_queries.csv")
print("2. knowledge_base.json")
print("3. tickets.csv")
print("4. audit_log.csv")
print("5. notification_log.csv")
print("6. rejected_queries.csv")

INTELLIGENT CUSTOMER SUPPORT SYSTEM - EXECUTION
Approved knowledge-base articles: 4

1. QUERY INGESTION
Total records: 13
Valid records: 11
Rejected records: 2

2. CLASSIFICATION AND WORKFLOW ROUTING


,query_id,intent,complexity,priority,confidence,matched_article,route,ticket_id,status
0,Q001,Login / Access,Medium,P2,0,None,CREATE_TICKET_AND_ESCALATE,TKT-A34FFF7D,OPEN
1,Q002,Billing,Medium,P2,0,None,CREATE_TICKET_AND_ESCALATE,TKT-27EB7795,OPEN
2,Q003,Technical Error,High,P1,0,None,URGENT_HUMAN_ESCALATION,TKT-8BD2F54F,OPEN
3,Q004,Feature Request,Low,P4,0,None,CREATE_TICKET_AND_ESCALATE,TKT-1861ACF5,OPEN
4,Q006,Unresolved Issue,High,P2,0,None,TECHNICAL_HUMAN_REVIEW,TKT-25410094,OPEN
5,Q007,Billing,Medium,P2,0,None,CREATE_TICKET_AND_ESCALATE,TKT-5B00B55F,OPEN
6,Q008,General Information,Medium,P3,0,None,CREATE_TICKET_AND_ESCALATE,TKT-1F99C415,OPEN
7,Q009,General Information,Low,P3,0,None,CREATE_TICKET_AND_ESCALATE,TKT-0418DBCA,OPEN
8,Q010,Technical Error,High,P1,0,None,URGENT_HUMAN_ESCALATION,TKT-7FB25F08,OPEN
9,Q011,Login / Access,Medium,P2,0,None,CREATE_TICKET_AND_ESCALATE,TKT-3319EEAF,OPEN



3. HUMAN-IN-THE-LOOP REVIEW
Q001 -> APPROVE -> RESOLVED
Q002 -> EDIT -> RESOLVED
Q004 -> HOLD -> ON_HOLD
Q008 -> ESCALATE -> ESCALATED
Q009 -> RESOLVE -> RESOLVED

4. NOTIFICATION RETRY TEST
Simulated failures: 2
Final delivery status: SENT
Total attempts: 3
Permanent failure test: FAILED

5. SLA MONITORING
At-risk test: AT_RISK
Breach test: BREACHED

6. HUMAN REVIEW AUTHORIZATION TEST
Invalid action rejected: Unauthorized or invalid review action

7. FINAL TICKET LIFECYCLE REPORT


,ticket_id,query_id,intent,priority,route,assigned_team,status,reviewer_action
0,TKT-A34FFF7D,Q001,Login / Access,P2,CREATE_TICKET_AND_ESCALATE,None,IN_REVIEW,APPROVE
1,TKT-27EB7795,Q002,Billing,P2,CREATE_TICKET_AND_ESCALATE,None,IN_REVIEW,EDIT
2,TKT-8BD2F54F,Q003,Technical Error,P1,URGENT_HUMAN_ESCALATION,None,OPEN,
3,TKT-1861ACF5,Q004,Feature Request,P4,CREATE_TICKET_AND_ESCALATE,None,ON_HOLD,HOLD
4,TKT-25410094,Q006,Unresolved Issue,P2,TECHNICAL_HUMAN_REVIEW,None,OPEN,
5,TKT-5B00B55F,Q007,Billing,P2,CREATE_TICKET_AND_ESCALATE,None,OPEN,
6,TKT-1F99C415,Q008,General Information,P3,CREATE_TICKET_AND_ESCALATE,None,ESCALATED,ESCALATE
7,TKT-0418DBCA,Q009,General Information,P3,CREATE_TICKET_AND_ESCALATE,None,RESOLVED,RESOLVE
8,TKT-7FB25F08,Q010,Technical Error,P1,URGENT_HUMAN_ESCALATION,None,OPEN,
9,TKT-3319EEAF,Q011,Login / Access,P2,CREATE_TICKET_AND_ESCALATE,None,OPEN,



8. NOTIFICATION DELIVERY REPORT


,message_id,ticket_id,channel,event,attempt,status,message
0,MSG-C6DE7123,TKT-A34FFF7D,Email + Messaging,RESOLVED,1,SENT,Your support ticket has been resolved.
1,MSG-988D9CF6,TKT-27EB7795,Email + Messaging,RESOLVED,1,SENT,Your support ticket has been resolved.
2,MSG-0FCFFD97,TKT-1F99C415,Email + Messaging,ESCALATED,1,SENT,Your issue has been escalated to specialist su...
3,MSG-0DC3DEDD,TKT-0418DBCA,Email + Messaging,RESOLVED,1,SENT,Your support ticket has been resolved.
4,MSG-A3100205,TKT-A34FFF7D,Email + Messaging,CREATED,1,FAILED,Your support ticket has been created.
5,MSG-A3100205,TKT-A34FFF7D,Email + Messaging,CREATED,2,FAILED,Your support ticket has been created.
6,MSG-A3100205,TKT-A34FFF7D,Email + Messaging,CREATED,3,SENT,Your support ticket has been created.
7,MSG-92F3E7DF,TKT-A34FFF7D,Email + Messaging,ESCALATED,1,FAILED,Your issue has been escalated to specialist su...
8,MSG-92F3E7DF,TKT-A34FFF7D,Email + Messaging,ESCALATED,2,FAILED,Your issue has been escalated to specialist su...
9,MSG-92F3E7DF,TKT-A34FFF7D,Email + Messaging,ESCALATED,3,FAILED,Your issue has been escalated to specialist su...


... (3 outputs left out)

,query_id,errors
0,Q005,Missing/empty query_text
1,Q001,Duplicate query ID



11. SYSTEM EVALUATION SUMMARY


,Evaluation Metric,Result
0,Intent Classification Accuracy,90.91%
1,Knowledge Base Resolution Rate,0.00%
2,Human Escalation / Review Rate,36.36%
3,Notification Success Rate,85.71%
4,SLA Compliance,50.00%
5,Valid Queries,11
6,Rejected Queries,2
7,Total Tickets Created,11
8,Notification Requests,7
9,Retried Notification Attempts,4



EXPERIMENT 08 EXECUTION COMPLETED
Generated files:
1. customer_queries.csv
2. knowledge_base.json
3. tickets.csv
4. audit_log.csv
5. notification_log.csv
6. rejected_queries.csv
